importing libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

<h1>Dataset Loading</h1>

In [2]:
df = pd.read_csv('dataset/BVBRC_genome_amr.csv', low_memory=False)


In [3]:
print(df.shape)

(171000, 21)


In [4]:
df.head()


,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,...,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Computational Method,Computational Method Version,Computational Method Performance,Evidence,Source,PubMed
0,562,562.174730,Escherichia coli strain UMDUCS23,trimethoprim/sulfamethoxazole,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,SIR XGBoost Model,202503101.0,"F1 score: 0.94, CI[0.94, 0.95]",Computational Method,NaN,NaN
1,562,562.144628,Escherichia coli 10,ceftriaxone,NaN,<=0.25,<=,0.25,mg/L,MIC,...,NaN,NaN,CLSI,2018.0,NaN,NaN,NaN,Laboratory Method,NaN,36165686
2,562,562.529500,Escherichia coli upec-113,ertapenem,NaN,0.5,NaN,0.5,mg/L,NaN,...,NaN,NaN,NaN,NaN,MIC XGBoost Model,202503101.0,"W1 score: 0.77, CI[0.66, 0.89]",Computational Method,NaN,NaN
3,562,562.178148,Escherichia coli PNUSAE022600,aztreonam,NaN,4.0,NaN,4.0,mg/L,NaN,...,NaN,BVBRC,NaN,NaN,MIC XGBoost Model,20250225.0,"W1 score: 0.61, CI[0.5, 0.73]",Computational Method,NaN,NaN
4,562,562.226510,Escherichia coli O23:H16 strain ECO0238,tetracycline,Resistant,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,SIR XGBoost Model,202503101.0,"F1 score: 0.93, CI[0.89, 0.97]",Computational Method,NaN,NaN


In [5]:
df.columns

Index(['Taxon ID', 'Genome ID', 'Genome Name', 'Antibiotic',
       'Resistant Phenotype', 'Measurement', 'Measurement Sign',
       'Measurement Value', 'Measurement Unit', 'Laboratory Typing Method',
       'Laboratory Typing Method Version', 'Laboratory Typing Platform',
       'Vendor', 'Testing Standard', 'Testing Standard Year',
       'Computational Method', 'Computational Method Version',
       'Computational Method Performance', 'Evidence', 'Source', 'PubMed'],
      dtype='object')

In [6]:
df.dtypes

Taxon ID                              int64
Genome ID                           float64
Genome Name                          object
Antibiotic                           object
Resistant Phenotype                  object
Measurement                          object
Measurement Sign                     object
Measurement Value                    object
Measurement Unit                     object
Laboratory Typing Method             object
Laboratory Typing Method Version     object
Laboratory Typing Platform           object
Vendor                               object
Testing Standard                     object
Testing Standard Year               float64
Computational Method                 object
Computational Method Version        float64
Computational Method Performance     object
Evidence                             object
Source                               object
PubMed                               object
dtype: object

In [7]:
df.describe()

,Taxon ID,Genome ID,Testing Standard Year,Computational Method Version
count,1.710000e+05,1.710000e+05,3701.000000,1.628280e+05
mean,4.943936e+04,4.865374e+04,2019.705755,1.656771e+08
std,2.514143e+05,2.493729e+05,2.044014,7.318143e+07
min,5.620000e+02,5.621000e+02,2008.000000,2.025022e+07
25%,5.620000e+02,5.621468e+02,2018.000000,2.025031e+08
50%,5.620000e+02,5.622211e+02,2021.000000,2.025031e+08
75%,5.620000e+02,5.627044e+02,2021.000000,2.025031e+08
max,3.050630e+06,3.050631e+06,2021.000000,2.025031e+08


<h1>data cleaning and feature engineering</h1>

handling target variable na values and unrelated outputs

In [8]:
df['Resistant Phenotype'].isna().sum()

np.int64(87829)

In [9]:
df['Resistant Phenotype'].unique()

array(['Susceptible', nan, 'Resistant', 'Intermediate', 'Nonsusceptible'],
      dtype=object)

In [10]:
df['Resistant Phenotype'].value_counts()

Resistant Phenotype
Susceptible       57912
Resistant         25183
Intermediate         75
Nonsusceptible        1
Name: count, dtype: int64

In [11]:
df = df.dropna(subset=['Resistant Phenotype'])
df.shape

(83171, 21)

In [12]:
df = df[df['Resistant Phenotype'].isin(['Resistant','Susceptible'])]
df.shape

(83095, 21)

In [13]:
print(df['Resistant Phenotype'].value_counts())

Resistant Phenotype
Susceptible    57912
Resistant      25183
Name: count, dtype: int64


handling duplicates

In [14]:
df.duplicated().sum()

np.int64(1)

In [15]:
df = df.drop_duplicates()
df.shape

(83094, 21)

null treatment

In [16]:
df.isna().sum()

Taxon ID                                0
Genome ID                               0
Genome Name                             0
Antibiotic                              0
Resistant Phenotype                     0
Measurement                         82390
Measurement Sign                    82527
Measurement Value                   82389
Measurement Unit                    82389
Laboratory Typing Method            80090
Laboratory Typing Method Version    83085
Laboratory Typing Platform          82024
Vendor                              65711
Testing Standard                    81096
Testing Standard Year               82527
Computational Method                 3099
Computational Method Version         3099
Computational Method Performance     3099
Evidence                               86
Source                              83094
PubMed                              81293
dtype: int64

In [17]:
df.shape

(83094, 21)

deleting columns having less null values greater than the threshold

In [18]:
threshold = 0.7 * len(df)

print("maximum null values a column can contains: " , threshold)


maximum null values a column can contains:  58165.799999999996


In [19]:
df = df.dropna(axis=1, thresh=threshold)

df.isna().sum()

Taxon ID                               0
Genome ID                              0
Genome Name                            0
Antibiotic                             0
Resistant Phenotype                    0
Computational Method                3099
Computational Method Version        3099
Computational Method Performance    3099
Evidence                              86
dtype: int64

dropping the computational method as it contain one one value that didn't give any information for the output

In [20]:
df['Computational Method'].value_counts()

Computational Method
SIR XGBoost Model    79995
Name: count, dtype: int64

In [21]:
df = df.drop(columns=['Computational Method'])

In [22]:
df.isna().sum()

Taxon ID                               0
Genome ID                              0
Genome Name                            0
Antibiotic                             0
Resistant Phenotype                    0
Computational Method Version        3099
Computational Method Performance    3099
Evidence                              86
dtype: int64

removing taxon id because it is predictable from the genome id adn cramer's V show 99.5 percent correlation and theoritcally genome id belong to a single taxon id

In [23]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    return np.sqrt(chi2 / (n * (min(confusion_matrix.shape)-1)))

v = cramers_v(df['Taxon ID'], df['Genome ID'])
print(f"Cramer's V: {v}")

Cramer's V: 0.9969007913454884


In [24]:
df = df.drop(columns=['Taxon ID'])
df.isna().sum()

Genome ID                              0
Genome Name                            0
Antibiotic                             0
Resistant Phenotype                    0
Computational Method Version        3099
Computational Method Performance    3099
Evidence                              86
dtype: int64

Handling genome name and genome id issue of count and droping genome id and also making the genome name consistent

In [25]:
print("genome id count: ", df['Genome ID'].value_counts().sum())
print("genome name count: ",df['Genome Name'].value_counts().sum())

genome id count:  83094
genome name count:  83094


In [26]:
df = df.drop(columns=['Genome ID'])
df.isna().sum()

Genome Name                            0
Antibiotic                             0
Resistant Phenotype                    0
Computational Method Version        3099
Computational Method Performance    3099
Evidence                              86
dtype: int64

In [27]:
df['Genome Name'] = df['Genome Name'].str.strip().str.strip("'")

..

In [28]:
df.isna().sum()

Genome Name                            0
Antibiotic                             0
Resistant Phenotype                    0
Computational Method Version        3099
Computational Method Performance    3099
Evidence                              86
dtype: int64

Dropping 'Computational Method Version' and 'Computational Method Performance' and 'evidence'
REASONING: 
1. These are metadata about the PREVIOUS model used to generate labels, 
    not biological traits of the bacteria. 
2. Including them would cause "Data Leakage," where the model learns 
    software versions instead of antimicrobial resistance patterns.
3. High cardinality in 'Version' causes unnecessary memory overhead.

In [29]:
df = df.drop(columns=['Computational Method Version', 'Computational Method Performance', 'Evidence'])

In [30]:
df.isna().sum()

Genome Name            0
Antibiotic             0
Resistant Phenotype    0
dtype: int64

<h1>Encodings</h1>

In [31]:
df

,Genome Name,Antibiotic,Resistant Phenotype
0,Escherichia coli strain UMDUCS23,trimethoprim/sulfamethoxazole,Susceptible
4,Escherichia coli O23:H16 strain ECO0238,tetracycline,Resistant
5,Escherichia coli strain 36549,cefalothin,Susceptible
6,Escherichia coli PNUSAE020616,norfloxacin,Susceptible
8,Escherichia coli strain 113,doxycycline,Resistant
...,...,...,...
170992,Escherichia coli strain 603388,cefazolin,Susceptible
170993,Escherichia coli 178200,cefotaxime,Susceptible
170994,Escherichia coli,levofloxacin,Susceptible
170995,Escherichia coli VREC1589,cefepime,Susceptible
